In [1]:
# 04_performance_benchmark.ipynb
# Estimation layer (RQ2 / H2): logistic-WoE vs stump-boosting on the SAME WoE grid,
# with unconstrained XGBoost as a reference ceiling (interpretability dropped).
# Metrics: discrimination (AUC, KS, Gini), stability (PSI train->test),
# sparsity/interpretability (n features, scorecard cells). 50 repeated stratified splits.
# Significance: DeLong test on paired AUC per split; Wilcoxon across splits.
# H2: stump-boosting is NOT statistically distinguishable from logistic-WoE in AUC
#     (superiority not assumed; tested per data).
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import xgboost as xgb
warnings.filterwarnings('ignore')

SEED = 42
N_SPLIT = 50
sns.set_theme(style='whitegrid', context='paper')
sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor': '0.2', 'grid.color': '0.85', 'font.size': 10})

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA   = os.path.join(ROOT, 'data')
TABLES = os.path.join(ROOT, 'results', 'tables')
FIGS   = os.path.join(ROOT, 'results', 'figures')

def savefig(fig, name):
    for ext in ('png','pdf'):
        fig.savefig(os.path.join(FIGS, name+'.'+ext), dpi=600, bbox_inches='tight')
    plt.close(fig)

# ===================================================================
# 0. LOAD
# ===================================================================
woe_X = pd.read_parquet(os.path.join(DATA, 'gmsc_woe.parquet'))
dfc   = pd.read_parquet(os.path.join(DATA, 'gmsc_clean.parquet'))
meta  = json.load(open(os.path.join(DATA, 'woe_maps.json')))
TARGET = meta['target']; FEATURES = meta['features']; MAPS = meta['maps']
y = woe_X[TARGET].values.astype(int)
Xw = woe_X[FEATURES].values                      # WoE design (logit + stump share this)
Xraw = dfc[FEATURES].fillna(dfc[FEATURES].median(numeric_only=True)).values  # xgb raw
n_cells = sum(sum(1 for r in MAPS[c] if r['bin']>=0) for c in FEATURES)
print('Data:', Xw.shape, '| features:', len(FEATURES), '| scorecard cells:', n_cells)

# ===================================================================
# 1. METRIC HELPERS
# ===================================================================
def ks_stat(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(np.max(tpr - fpr))

def psi(expected, actual, bins=10):
    """Population Stability Index of a score between train(expected) and test(actual)."""
    qs = np.quantile(expected, np.linspace(0,1,bins+1))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.histogram(expected, qs)[0] / len(expected)
    a = np.histogram(actual,   qs)[0] / len(actual)
    e = np.clip(e, 1e-6, None); a = np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

def delong_var_auc_diff(y_true, s1, s2):
    """DeLong test: z, p for AUC(s1) - AUC(s2) on same sample (paired)."""
    y_true = np.asarray(y_true)
    pos = y_true == 1; neg = ~pos
    m, nn = pos.sum(), neg.sum()
    def structural(s):
        sp, sn = s[pos], s[neg]
        # midrank-based placements (Sun & Xu / DeLong)
        V10 = np.array([ (np.sum(sn < x) + 0.5*np.sum(sn == x))/nn for x in sp ])
        V01 = np.array([ (np.sum(sp > x) + 0.5*np.sum(sp == x))/m  for x in sn ])
        auc = V10.mean()
        return auc, V10, V01
    a1,V10_1,V01_1 = structural(s1); a2,V10_2,V01_2 = structural(s2)
    S10 = np.cov(np.vstack([V10_1, V10_2]))
    S01 = np.cov(np.vstack([V01_1, V01_2]))
    S = S10/m + S01/nn
    var = S[0,0] + S[1,1] - 2*S[0,1]
    if var <= 0: return a1, a2, np.nan, np.nan
    z = (a1 - a2)/np.sqrt(var)
    p = 2*(1 - stats.norm.cdf(abs(z)))
    return a1, a2, float(z), float(p)

# ===================================================================
# 2. REPEATED SPLITS
# ===================================================================
recs = []
delong_rows = []
t0 = time.time()
for sp in range(N_SPLIT):
    Xw_tr,Xw_te,Xr_tr,Xr_te,ytr,yte = train_test_split(
        Xw, Xraw, y, test_size=0.3, stratify=y, random_state=SEED+sp)
    # logistic-WoE
    lr = LogisticRegression(max_iter=1000).fit(Xw_tr, ytr)
    s_lr_tr = lr.decision_function(Xw_tr); s_lr = lr.decision_function(Xw_te)
    # stump-boosting (same WoE grid, multivariate)
    gb = GradientBoostingClassifier(max_depth=1, learning_rate=0.3,
                                    n_estimators=300, random_state=SEED+sp).fit(Xw_tr, ytr)
    s_gb_tr = gb.decision_function(Xw_tr); s_gb = gb.decision_function(Xw_te)
    # xgboost ceiling (raw features, unconstrained)
    xg = xgb.XGBClassifier(max_depth=4, n_estimators=300, learning_rate=0.1,
                           subsample=0.9, colsample_bytree=0.9, n_jobs=4,
                           eval_metric='logloss', random_state=SEED+sp).fit(Xr_tr, ytr)
    s_xg = xg.predict_proba(Xr_te)[:,1]; s_xg_tr = xg.predict_proba(Xr_tr)[:,1]
    for name, s_te, s_tr in [('logit_woe',s_lr,s_lr_tr),
                             ('stump_boost',s_gb,s_gb_tr),
                             ('xgb_ceiling',s_xg,s_xg_tr)]:
        recs.append([sp, name, roc_auc_score(yte,s_te), ks_stat(yte,s_te),
                     2*roc_auc_score(yte,s_te)-1, psi(s_tr,s_te)])
    # DeLong: stump vs logit on the same test fold
    a_gb,a_lr,z,p = delong_var_auc_diff(yte, s_gb, s_lr)
    delong_rows.append([sp, a_gb, a_lr, a_gb-a_lr, z, p])
    if (sp+1) % 10 == 0:
        print('  split %d/%d (%.0fs)' % (sp+1, N_SPLIT, time.time()-t0))
print('Splits done in %.0fs' % (time.time()-t0))

perf = pd.DataFrame(recs, columns=['split','model','auc','ks','gini','psi'])
delong = pd.DataFrame(delong_rows, columns=['split','auc_stump','auc_logit','auc_diff','z','p'])

# ===================================================================
# 3. SUMMARY + H2 TEST
# ===================================================================
summ = perf.groupby('model').agg(
    auc_mean=('auc','mean'), auc_std=('auc','std'),
    ks_mean=('ks','mean'),  ks_std=('ks','std'),
    gini_mean=('gini','mean'), psi_mean=('psi','mean')).reindex(
    ['logit_woe','stump_boost','xgb_ceiling'])
# sparsity/interpretability
summ['n_features'] = [len(FEATURES), len(FEATURES), len(FEATURES)]
summ['scorecard_cells'] = [n_cells, n_cells, np.nan]
print('\n=== Performance summary (50 splits) ===')
print(summ.round(4).to_string())

# paired comparison stump vs logit across splits
auc_l = perf[perf.model=='logit_woe'].set_index('split')['auc']
auc_s = perf[perf.model=='stump_boost'].set_index('split')['auc']
auc_x = perf[perf.model=='xgb_ceiling'].set_index('split')['auc']
w_sl = stats.wilcoxon(auc_s, auc_l)
print('\n=== H2: stump-boost vs logit-WoE (AUC) ===')
print('mean AUC diff (stump - logit) = %+.5f' % (auc_s.mean()-auc_l.mean()))
print('Wilcoxon across splits: stat=%.1f p=%.4f' % (w_sl.statistic, w_sl.pvalue))
print('DeLong per-split: mean z=%.3f, median p=%.4f, frac p<0.05 = %.2f'
      % (delong['z'].mean(), delong['p'].median(), (delong['p']<0.05).mean()))
print('Gap to ceiling: logit %+.5f, stump %+.5f (AUC - xgb)'
      % (auc_l.mean()-auc_x.mean(), auc_s.mean()-auc_x.mean()))

# ===================================================================
# 4. SAVE TABLES
# ===================================================================
perf.round(6).to_csv(os.path.join(TABLES, '04_perf_long.csv'), index=False)
summ.round(6).to_csv(os.path.join(TABLES, '04_perf_summary.csv'))
delong.round(6).to_csv(os.path.join(TABLES, '04_delong_per_split.csv'), index=False)

# ===================================================================
# 5. FIGURE A — AUC & KS distributions across splits (box, greyscale)
# ===================================================================
order = ['logit_woe','stump_boost','xgb_ceiling']
lbl = {'logit_woe':'logistic-WoE','stump_boost':'stump-boost','xgb_ceiling':'XGBoost (ceiling)'}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
greys = ['0.35','0.55','0.8']
for ax, metric, ttl in [(axes[0],'auc','AUC'), (axes[1],'ks','KS')]:
    data = [perf[perf.model==m][metric].values for m in order]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6,
                    medianprops=dict(color='0.0', lw=1.3),
                    flierprops=dict(marker='o', markersize=3, markerfacecolor='0.5', markeredgecolor='0.3'))
    for patch, c in zip(bp['boxes'], greys):
        patch.set_facecolor(c); patch.set_edgecolor('0.0')
    ax.set_xticklabels([lbl[m] for m in order], rotation=15)
    ax.set_ylabel(ttl); ax.set_title('%s across %d splits' % (ttl, N_SPLIT), fontsize=10)
savefig(fig, '04_auc_ks_box')

# ===================================================================
# 6. FIGURE B — paired AUC diff (stump - logit) per split with 0 line
# ===================================================================
fig, ax = plt.subplots(figsize=(7.2, 4.2))
d = delong['auc_diff'].values
ax.axhline(0, color='0.0', lw=1, ls='--')
ax.scatter(np.arange(len(d)), d, s=34, c='0.4', edgecolor='0.0', linewidth=.4)
ax.axhline(d.mean(), color='0.2', lw=1.2, label='mean = %+.5f' % d.mean())
ax.set_xlabel('split'); ax.set_ylabel('AUC(stump) \u2212 AUC(logit)')
ax.legend(frameon=True, facecolor='white', edgecolor='0.5')
savefig(fig, '04_auc_diff_per_split')

print('\nSaved figures: 04_auc_ks_box, 04_auc_diff_per_split')
print('Saved tables : 04_perf_long, 04_perf_summary, 04_delong_per_split')

# ===================================================================
# 7. HEADLINE
# ===================================================================
print('\n================ RQ2 / H2 HEADLINE ================')
print('logit-WoE  AUC = %.4f \u00b1 %.4f' % (auc_l.mean(), auc_l.std()))
print('stump-boost AUC = %.4f \u00b1 %.4f' % (auc_s.mean(), auc_s.std()))
print('XGB ceiling AUC = %.4f \u00b1 %.4f' % (auc_x.mean(), auc_x.std()))
verdict = ('indistinguishable' if w_sl.pvalue > 0.05 else
           ('stump higher' if auc_s.mean()>auc_l.mean() else 'logit higher'))
print('H2 verdict (Wilcoxon p=%.3f): %s' % (w_sl.pvalue, verdict))
print('\n04 complete.')


Data: (150000, 7) | features: 7 | scorecard cells: 30


  split 10/50 (92s)


  split 20/50 (181s)


  split 30/50 (269s)


  split 40/50 (358s)


  split 50/50 (448s)
Splits done in 448s

=== Performance summary (50 splits) ===
             auc_mean  auc_std  ks_mean  ks_std  gini_mean  psi_mean  n_features  scorecard_cells
model                                                                                            
logit_woe      0.8461   0.0031   0.5420  0.0066     0.6922    0.0003           7             30.0
stump_boost    0.8473   0.0031   0.5432  0.0063     0.6945    0.0003           7             30.0
xgb_ceiling    0.8520   0.0032   0.5511  0.0071     0.7041    0.0003           7              NaN

=== H2: stump-boost vs logit-WoE (AUC) ===
mean AUC diff (stump - logit) = +0.00119
Wilcoxon across splits: stat=0.0 p=0.0000
DeLong per-split: mean z=3.007, median p=0.0051, frac p<0.05 = 0.90
Gap to ceiling: logit -0.00595, stump -0.00476 (AUC - xgb)



Saved figures: 04_auc_ks_box, 04_auc_diff_per_split
Saved tables : 04_perf_long, 04_perf_summary, 04_delong_per_split

================ RQ2 / H2 HEADLINE ================
logit-WoE  AUC = 0.8461 ± 0.0031
stump-boost AUC = 0.8473 ± 0.0031
XGB ceiling AUC = 0.8520 ± 0.0032
H2 verdict (Wilcoxon p=0.000): stump higher

04 complete.
